In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import time

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    log_loss
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

import lightgbm as lgb
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', 200)
plt.rcParams['figure.dpi'] = 100

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("loan-default")

print(f"LightGBM: {lgb.__version__}")

In [ ]:
df = pd.read_parquet('../data/loans_clean.parquet')

df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df['issue_year'] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

GAIN_RATE = 0.10
LOSS_RATE = 0.50

def calculate_profit(df_subset, approval_mask, gain_rate=GAIN_RATE, loss_rate=LOSS_RATE):
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved['default'] == 0,
         gain_rate * approved['loan_amnt'],
        -loss_rate * approved['loan_amnt']
    )
    return profit.sum()

In [ ]:
def prep_features(df):
    df = df.copy()
    redundant = ['fico_range_high', 'funded_amnt', 'funded_amnt_inv',
                 'num_sats', 'installment', 'num_rev_tl_bal_gt_0']
    joint_cols = [c for c in df.columns if c.startswith('sec_app_') or c.endswith('_joint')]
    high_cardinality = ['zip_code', 'sub_grade']
    split_cols = ['issue_year']
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    emp_map = {f'{i} year{"s" if i != 1 else ""}': i for i in range(1, 10)}
    emp_map['< 1 year'] = 0
    emp_map['10+ years'] = 10
    df['emp_length'] = df['emp_length'].map(emp_map)
    return df


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ]
    )


df_train_fe = prep_features(df_train)
df_val_fe   = prep_features(df_val)

X_train = df_train_fe.drop(columns=['default'])
X_val   = df_val_fe.drop(columns=['default'])

X_train['term'] = X_train['term'].str.extract(r'(\d+)').astype(int)
X_val['term']   = X_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)

print(f"Train shape: {X_train_proc.shape}")
print(f"Val shape:   {X_val_proc.shape}")

In [ ]:
# LogReg (the model with bad calibration)
print("Training LogReg...")
logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    n_jobs=-1
)
logreg.fit(X_train_proc, y_train)
logreg_probs = logreg.predict_proba(X_val_proc)[:, 1]
print(f"  AUC: {roc_auc_score(y_val, logreg_probs):.4f}")
print(f"  Brier: {brier_score_loss(y_val, logreg_probs):.4f}")

# LightGBM tuned (the champion)
print("\nTraining LightGBM tuned...")
best_params = {
    'learning_rate': 0.021624893900603594,
    'num_leaves': 58,
    'min_child_samples': 83,
    'feature_fraction': 0.5687015336457729,
    'bagging_fraction': 0.6662082814499004,
    'bagging_freq': 3,
    'lambda_l1': 0.003248711401116576,
    'lambda_l2': 9.48498230595779,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

lgbm = LGBMClassifier(**best_params)
lgbm.fit(
    X_train_proc, y_train,
    eval_set=[(X_val_proc, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)
lgbm_probs = lgbm.predict_proba(X_val_proc)[:, 1]
print(f"  AUC: {roc_auc_score(y_val, lgbm_probs):.4f}")
print(f"  Brier: {brier_score_loss(y_val, lgbm_probs):.4f}")
print(f"  Iterations: {lgbm.best_iteration_}")


In [ ]:
from sklearn.calibration import calibration_curve

# Calibration curves use 10 quantile bins so each bin has roughly equal sample count
prob_true_lr, prob_pred_lr = calibration_curve(y_val, logreg_probs, n_bins=10, strategy='quantile')
prob_true_lgbm, prob_pred_lgbm = calibration_curve(y_val, lgbm_probs, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LogReg
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred_lr, prob_true_lr, 's-', color='#d62728', linewidth=2, markersize=8,
        label=f'LogReg (Brier {brier_score_loss(y_val, logreg_probs):.4f})')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual default rate')
ax.set_title('Logistic Regression — uncalibrated')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# LightGBM
ax = axes[1]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred_lgbm, prob_true_lgbm, 's-', color='#1f77b4', linewidth=2, markersize=8,
        label=f'LightGBM tuned (Brier {brier_score_loss(y_val, lgbm_probs):.4f})')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual default rate')
ax.set_title('LightGBM tuned — uncalibrated')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/calibration_before.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Split val 50/50: half to fit the calibrator, half to evaluate calibrated performance
np.random.seed(42)
val_size = len(X_val_proc)
calib_idx = np.random.choice(val_size, size=val_size//2, replace=False)
eval_idx = np.setdiff1d(np.arange(val_size), calib_idx)

X_val_calib = X_val_proc[calib_idx]
y_val_calib = y_val[calib_idx]
X_val_eval  = X_val_proc[eval_idx]
y_val_eval  = y_val[eval_idx]

print(f"Calibration set: {len(calib_idx):,} samples")
print(f"Evaluation set:  {len(eval_idx):,} samples")

# Get raw uncalibrated probabilities
logreg_probs_calib = logreg.predict_proba(X_val_calib)[:, 1]
logreg_probs_eval  = logreg.predict_proba(X_val_eval)[:, 1]
lgbm_probs_calib   = lgbm.predict_proba(X_val_calib)[:, 1]
lgbm_probs_eval    = lgbm.predict_proba(X_val_eval)[:, 1]

# Fit isotonic calibrators on calib half
print("\nFitting isotonic calibrators...")
iso_logreg = IsotonicRegression(out_of_bounds='clip')
iso_logreg.fit(logreg_probs_calib, y_val_calib)

iso_lgbm = IsotonicRegression(out_of_bounds='clip')
iso_lgbm.fit(lgbm_probs_calib, y_val_calib)

# Apply calibrators to eval half probabilities
logreg_cal_eval = iso_logreg.predict(logreg_probs_eval)
lgbm_cal_eval   = iso_lgbm.predict(lgbm_probs_eval)

# Compute metrics on eval half: before vs after calibration
logreg_uncal_eval = logreg_probs_eval  # alias for clarity
lgbm_uncal_eval   = lgbm_probs_eval

print("\n=== CALIBRATION RESULTS (evaluated on held-out half of val) ===\n")
print("LogReg:")
print(f"  Before  AUC: {roc_auc_score(y_val_eval, logreg_uncal_eval):.4f}  Brier: {brier_score_loss(y_val_eval, logreg_uncal_eval):.4f}")
print(f"  After   AUC: {roc_auc_score(y_val_eval, logreg_cal_eval):.4f}  Brier: {brier_score_loss(y_val_eval, logreg_cal_eval):.4f}")
brier_lr_before = brier_score_loss(y_val_eval, logreg_uncal_eval)
brier_lr_after = brier_score_loss(y_val_eval, logreg_cal_eval)
print(f"  Brier improvement: {(brier_lr_before - brier_lr_after) / brier_lr_before * 100:.1f}%")

print("\nLightGBM tuned:")
print(f"  Before  AUC: {roc_auc_score(y_val_eval, lgbm_uncal_eval):.4f}  Brier: {brier_score_loss(y_val_eval, lgbm_uncal_eval):.4f}")
print(f"  After   AUC: {roc_auc_score(y_val_eval, lgbm_cal_eval):.4f}  Brier: {brier_score_loss(y_val_eval, lgbm_cal_eval):.4f}")
brier_lg_before = brier_score_loss(y_val_eval, lgbm_uncal_eval)
brier_lg_after = brier_score_loss(y_val_eval, lgbm_cal_eval)
print(f"  Brier improvement: {(brier_lg_before - brier_lg_after) / brier_lg_before * 100:.1f}%")

In [ ]:
# Compute calibration curves on the eval half (apples-to-apples comparison)
prob_true_lr_b, prob_pred_lr_b = calibration_curve(y_val_eval, logreg_uncal_eval, n_bins=10, strategy='quantile')
prob_true_lr_a, prob_pred_lr_a = calibration_curve(y_val_eval, logreg_cal_eval, n_bins=10, strategy='quantile')
prob_true_lg_b, prob_pred_lg_b = calibration_curve(y_val_eval, lgbm_uncal_eval, n_bins=10, strategy='quantile')
prob_true_lg_a, prob_pred_lg_a = calibration_curve(y_val_eval, lgbm_cal_eval, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LogReg before/after
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred_lr_b, prob_true_lr_b, 's-', color='#d62728', linewidth=2, markersize=8,
        label=f'Before (Brier {brier_score_loss(y_val_eval, logreg_uncal_eval):.4f})')
ax.plot(prob_pred_lr_a, prob_true_lr_a, 'o-', color='#2ca02c', linewidth=2, markersize=8,
        label=f'After (Brier {brier_score_loss(y_val_eval, logreg_cal_eval):.4f})')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual default rate')
ax.set_title('LogReg: isotonic calibration')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# LightGBM before/after
ax = axes[1]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred_lg_b, prob_true_lg_b, 's-', color='#d62728', linewidth=2, markersize=8,
        label=f'Before (Brier {brier_score_loss(y_val_eval, lgbm_uncal_eval):.4f})')
ax.plot(prob_pred_lg_a, prob_true_lg_a, 'o-', color='#2ca02c', linewidth=2, markersize=8,
        label=f'After (Brier {brier_score_loss(y_val_eval, lgbm_cal_eval):.4f})')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual default rate')
ax.set_title('LightGBM tuned: isotonic calibration')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/calibration_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Use calibrated LightGBM probabilities and the val_eval subset
df_val_eval = df_val.iloc[eval_idx].reset_index(drop=True)

# Compute profit at fine-grained threshold steps
thresholds_full = np.linspace(0.01, 0.55, 200)
profits_full = np.array([calculate_profit(df_val_eval, lgbm_cal_eval < t) for t in thresholds_full])

# Find optimum
best_idx = int(np.argmax(profits_full))
best_threshold = thresholds_full[best_idx]
best_profit = profits_full[best_idx]

# Theoretical break-even (where expected profit per loan = 0)
break_even = GAIN_RATE / (GAIN_RATE + LOSS_RATE)

print(f"Optimal threshold:   {best_threshold:.4f}")
print(f"Optimal profit:      ${best_profit:,.0f}")
print(f"Break-even (theory): {break_even:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds_full, profits_full/1e6, color='#1f77b4', linewidth=2.5)
ax.axvline(best_threshold, color='#2ca02c', linestyle='--', alpha=0.7,
           label=f'Empirical optimum ({best_threshold:.3f})')
ax.axvline(break_even, color='#d62728', linestyle=':', alpha=0.7,
           label=f'Theoretical break-even ({break_even:.3f})')
ax.axhline(0, color='black', alpha=0.3, linewidth=0.5)
ax.scatter([best_threshold], [best_profit/1e6], color='#2ca02c', s=120, zorder=5)
ax.set_xlabel('Threshold (approve if P(default) < threshold)')
ax.set_ylabel('Profit on val_eval half ($M)')
ax.set_title('Profit curve — calibrated LightGBM tuned')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/profit_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Walk along thresholds, compute approval rate and default rate among approved
approval_rates = []
default_rates_approved = []
profits_at_rate = []

for t in thresholds_full:
    approval_mask = lgbm_cal_eval < t
    if approval_mask.sum() > 0:
        approval_rates.append(approval_mask.mean())
        default_rates_approved.append(df_val_eval[approval_mask]['default'].mean())
        profits_at_rate.append(calculate_profit(df_val_eval, approval_mask))
    else:
        approval_rates.append(0)
        default_rates_approved.append(0)
        profits_at_rate.append(0)

approval_rates = np.array(approval_rates)
default_rates_approved = np.array(default_rates_approved)
profits_at_rate = np.array(profits_at_rate)

# Plot with dual y-axes
fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = '#d62728'
ax1.set_xlabel('Approval rate')
ax1.set_ylabel('Default rate among approved', color=color1)
ax1.plot(approval_rates, default_rates_approved, color=color1, linewidth=2.5,
         label='Default rate among approved')
ax1.axhline(break_even, color=color1, linestyle=':', alpha=0.5,
            label=f'Break-even default rate ({break_even:.3f})')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 1)

ax2 = ax1.twinx()
color2 = '#2ca02c'
ax2.set_ylabel('Profit ($M)', color=color2)
ax2.plot(approval_rates, profits_at_rate/1e6, color=color2, linewidth=2.5, label='Profit')
ax2.axhline(0, color='black', alpha=0.3, linewidth=0.5)
ax2.tick_params(axis='y', labelcolor=color2)

opt_approval_rate = (lgbm_cal_eval < best_threshold).mean()
ax1.axvline(opt_approval_rate, color='#1f77b4', linestyle='--', alpha=0.5,
            label=f'Optimal approval rate ({opt_approval_rate:.1%})')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.title('Tradeoff: more approvals → higher default rate, profit eventually turns negative')
plt.tight_layout()
plt.savefig('../reports/approval_default_tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# What if we got the cost matrix wrong? Sweep across plausible gain/loss combinations
gain_rates = np.linspace(0.05, 0.20, 8)
loss_rates = np.linspace(0.30, 0.70, 9)

opt_profit_matrix = np.zeros((len(loss_rates), len(gain_rates)))
opt_threshold_matrix = np.zeros((len(loss_rates), len(gain_rates)))

print("Computing sensitivity matrix...")
for i, lr in enumerate(loss_rates):
    for j, gr in enumerate(gain_rates):
        profits = np.array([calculate_profit(df_val_eval, lgbm_cal_eval < t, gain_rate=gr, loss_rate=lr) 
                            for t in thresholds_full])
        opt_idx = int(np.argmax(profits))
        opt_profit_matrix[i, j] = profits[opt_idx]
        opt_threshold_matrix[i, j] = thresholds_full[opt_idx]

# Plot heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
im = ax.imshow(opt_profit_matrix/1e6, cmap='RdYlGn', aspect='auto', origin='lower',
               extent=[gain_rates[0], gain_rates[-1], loss_rates[0], loss_rates[-1]])
ax.set_xlabel('GAIN_RATE (return on good loans)')
ax.set_ylabel('LOSS_RATE (loss rate on defaults)')
ax.set_title('Optimal profit ($M) across cost assumptions')
ax.scatter([GAIN_RATE], [LOSS_RATE], color='blue', s=300, marker='*', zorder=5,
           edgecolors='white', linewidth=2, label='Our base case')
ax.legend(loc='upper right')
plt.colorbar(im, ax=ax, label='Profit ($M)')

ax = axes[1]
im = ax.imshow(opt_threshold_matrix, cmap='viridis', aspect='auto', origin='lower',
               extent=[gain_rates[0], gain_rates[-1], loss_rates[0], loss_rates[-1]])
ax.set_xlabel('GAIN_RATE')
ax.set_ylabel('LOSS_RATE')
ax.set_title('Optimal threshold across cost assumptions')
ax.scatter([GAIN_RATE], [LOSS_RATE], color='red', s=300, marker='*', zorder=5,
           edgecolors='white', linewidth=2, label='Our base case')
ax.legend(loc='upper right')
plt.colorbar(im, ax=ax, label='Threshold')

plt.tight_layout()
plt.savefig('../reports/cost_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n=== SENSITIVITY ANALYSIS ===")
print(f"Base case (GAIN=0.10, LOSS=0.50): ${opt_profit_matrix[4, 3]/1e6:.1f}M")
print(f"Best case in sweep: ${opt_profit_matrix.max()/1e6:.1f}M")
print(f"Worst case in sweep: ${opt_profit_matrix.min()/1e6:.1f}M")
print(f"\nThreshold range across sweep: [{opt_threshold_matrix.min():.3f}, {opt_threshold_matrix.max():.3f}]")


In [ ]:
import pickle
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

# Save the calibrated LightGBM as a tuple: (preprocessor, base_model, isotonic_calibrator)
champion_artifact = {
    'preprocessor': preprocessor,
    'base_model': lgbm,
    'calibrator': iso_lgbm,
    'best_threshold': float(best_threshold),
    'best_params': best_params,
    'val_metrics': {
        'auc_uncalibrated': float(roc_auc_score(y_val_eval, lgbm_uncal_eval)),
        'auc_calibrated':   float(roc_auc_score(y_val_eval, lgbm_cal_eval)),
        'brier_uncalibrated': float(brier_score_loss(y_val_eval, lgbm_uncal_eval)),
        'brier_calibrated':   float(brier_score_loss(y_val_eval, lgbm_cal_eval)),
        'optimal_threshold': float(best_threshold),
        'optimal_profit_val_eval': float(best_profit),
    }
}

with open('../models/champion_lgbm.pkl', 'wb') as f:
    pickle.dump(champion_artifact, f)

print("Champion model saved to models/champion_lgbm.pkl")
print(f"Size: {Path('../models/champion_lgbm.pkl').stat().st_size / 1e6:.1f} MB")